# 第十章：非线性降维与生成模型

本章承接第九章的 PCA（线性降维），深入非线性降维方法（t-SNE/UMAP），并过渡到生成模型（Autoencoder → VAE → DDcGAN）。

## 10.1 t-SNE：看到 PCA 看不到的结构

PCA 是**线性**降维——它只能捕捉数据中的线性关系。当数据分布在高维空间中的
复杂曲面上时，PCA 会丢失重要结构。

**t-SNE (t-Distributed Stochastic Neighbor Embedding, 2008)** 是非线性降维的黄金标准。
它通过保持高维和低维空间中点对之间的**局部相似性**来揭示数据的内在聚类结构。

### PCA vs t-SNE

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **PCA** | 保留最大方差的线性方向 | 速度快，可解释 | 无法捕捉非线性结构 |
| **t-SNE** | 保持局部邻居相似性 | 揭示聚类，视觉漂亮 | 计算慢，随机性（不同运行结果不同） |


### 三种降维的本质区别

| 方法 | 优化目标 | 保留什么 | 复杂度 |
|------|---------|---------|--------|
| PCA | $\max 	ext{Var}$ | 全局线性方差 | (nd^2)$ |
| t-SNE | $\min KL(P\|Q)$ | 局部邻域 | (n^2)$ |
| UMAP | 模糊单纯形交叉熵 | 局部+部分全局 | (n^{1.14})$ |

t-SNE 高维用高斯核 {j|i} \propto \exp(-\|x_i-x_j\|^2/2\sigma_i^2)$，低维用 t 分布 {ij} \propto (1+\|y_i-y_j\|^2)^{-1}$。t 分布的重尾解决了拥挤问题——中距离点对在低维分散开。


### 10.1.1 t-SNE 的数学原理

t-SNE 的核心思想：在高维空间中相似的样本点，在低维空间中也应该靠近。

#### Step 1: 高维相似性（高斯核）

对每个样本 $x_i$，定义它与其他样本的条件概率（"$x_j$ 是 $x_i$ 的邻居"）：

$$

p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}

$$

$\sigma_i$ 由 **perplexity** 参数控制，使得每个样本的有效邻居数大致等于 perplexity。

对称化：$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$

#### Step 2: 低维嵌入（t 分布）

在低维空间中，用自由度为 1 的 Student t 分布（即 Cauchy 分布）来度量相似性：

$$

q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}

$$

#### Step 3: KL 散度最小化

t-SNE 最小化 $P$ 和 $Q$ 之间的 KL 散度：

$$

\mathcal{L} = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}

$$

#### 为什么用 t 分布而非高斯？

高斯分布的尾部太薄——中距离的点对在低维空间中要么挤在一起（无法区分），要么推得太远。t 分布的**重尾 (heavy tail)** 属性允许中距离点对在低维空间中适度分开，产生更好的聚类分离效果。

### 9.1.2 t-SNE vs UMAP：非线性降维的选择困境

| 维度 | t-SNE | UMAP |
|------|-------|------|
| 数学基础 | 概率（KL 散度） | 拓扑（模糊单纯形） |
| 全局结构 | 较差（距离无意义） | 较好（保留部分全局关系） |
| 速度 | $O(n^2)$ | $O(n^{1.14})$ |
| 可复现性 | 随机，不同运行结果不同 | 使用随机种子可复现 |
| 最佳场景 | 聚类可视化 | 大规模数据探索 |


In [ ]:
# === PCA vs t-SNE 对比 ===
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import time

# 生成复杂数据
np.random.seed(42)  # 固定随机种子，保证结果可复现
n_samples = 500

# 三维瑞士卷数据（Swiss Roll）
t = np.linspace(0, 15, n_samples)  # 生成等间隔序列
X_swiss = np.zeros((n_samples, 3))  # 创建全零数组
X_swiss[:, 0] = t * np.cos(t)
X_swiss[:, 1] = t * np.sin(t)
X_swiss[:, 2] = t
color = np.arctan2(X_swiss[:, 1], X_swiss[:, 0])

print("数据形状:", X_swiss.shape)

# PCA
t0 = time.time()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)  # 拟合并应用变换
t_pca = time.time() - t0
print(f"PCA: {X_pca.shape}, 用时 {t_pca:.3f}s")

# t-SNE (需要 scikit-learn)
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_swiss)  # 拟合并应用变换
t_tsne = time.time() - t0
print(f"t-SNE: {X_tsne.shape}, 用时 {t_tsne:.3f}s")

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(14, 5))  # 创建子图网格
methods = [('PCA', X_pca, 'PCA降维'), 
           ('t-SNE', X_tsne, 't-SNE降维'),
           ('Original', X_swiss, '原始3D数据 (仅第一维度着色)')]

for i, (title, data, label) in enumerate(methods):
    ax = axes[i]
    if data.shape[1] == 3:  # 3D数据
        ax.scatter(data[:, 0], data[:, 1], c=data[:, 2],
                   c=color, s=3, alpha=0.5)
        ax.set_title(f'{title}\n({label})')
    else:
        ax.scatter(data[:, 0], data[:, 1], c=color, cmap='viridis', s=5)
        ax.set_title(f'{title}\n({label})')
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
ax.legend(fontsize=8)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/pca_vs_tsne.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("\n关键观察:")
print(f"  PCA: 找到 {X_pca.shape[1]} 维 → 丢失了非线性结构")
print(f"  t-SNE: 找到 {X_tsne.shape[1]} 维 → 发现了 {len(np.unique(color))} 个聚类")


## 10.2 UMAP：更快更稳定的替代方案

**UMAP (Uniform Manifold Approximation and Projection, 2018)** 是比 t-SNE 更新的
非线性降维方法。它在保持全局结构方面优于 t-SNE，同时运行速度更快、结果可复现。

### t-SNE 的痛点

- **随机性**：不同运行给出不同结果
- **perplexity**：关键超参数，控制局部 vs 全局结构的平衡（典型值 5-50）
- **计算成本**：$O(n^2)$ 或更高

### 选择建议

- 需要**可复现**的结果：使用 UMAP
- 需要**探索性分析**：t-SNE 多次运行观察聚类稳定性
- 需要**速度**：PCA 用于快速预览


In [ ]:
# === 手写数字 t-SNE 可视化 ===
import matplotlib.pyplot as plt
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits 数据: {X_digits.shape}")

# PCA 快速预览
pca_digits = PCA(n_components=2).fit_transform(X_digits)  # 拟合并应用变换
print(f"PCA (2D): 前两个主成分")

# t-SNE
tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne_d = tsne_digits.fit_transform(X_digits)  # 拟合并应用变换
print(f"t-SNE (2D)")

# UMAP (如果可用)
try:
    import umap
    umap_reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = umap_reducer.fit_transform(X_digits)  # 拟合并应用变换
    print(f"UMAP (2D): {X_umap.shape}")
except ImportError:
    print("UMAP 未安装。安装: pip install umap-learn")

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 6))  # 创建子图网格
plot_data = [
    ('PCA', pca_digits),
    ('t-SNE', X_tsne_d),
]
try:
    plot_data.append(('UMAP', X_umap))
except NameError:
    pass

for i, (title, data) in enumerate(plot_data):
    ax = axes[i]
    scatter = ax.scatter(data[:, 0], data[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.6)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].legend(fontsize=8)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/tsne_umap_digits.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 10.3 自编码器 (Autoencoder)：生成模型的基石

自编码器是最简单的**生成模型**——它学习将输入复制到输出。
虽然看起来"什么都没做"，但这个"复制"任务迫使网络学习
数据的**紧凑表示 (compact representation)**。

### 自编码器架构

```
输入 x → [Encoder] → 潜在码 z → [Decoder] → 重建 x̂
```

### 数学定义

- **Encoder**: $\mathbf{z} = f_\theta(\mathbf{x})$ 将输入压缩为低维潜在表示
- **Decoder**: $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$ 从潜在码重建输入
- **损失函数**: $\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$

### 为什么需要自编码器？

1. **降维**：学到数据的紧凑表示 → 可用于可视化、压缩、去噪
2. **生成**：从潜在空间采样 → 可生成新数据
3. **异常检测**：重建误差大的样本 → 可能是异常值


### 10.3.1 潜在空间与流形假设

自编码器的核心产物是**潜在空间 (Latent Space)**——将高维数据压缩到的低维连续空间。

#### 流形假设 (Manifold Hypothesis)

真实世界的高维数据（如图像、声音）虽然表面上是高维的（一张 28×28 图片 = 784 维），但实际上分布在低维流形上——所有合法的 MNIST 手写数字只占据 784 维空间中的一个极小区域。

自编码器做的事情就是**学习这个流形的参数化**：Encoder 将流形上的点映射到潜在空间坐标，Decoder 将坐标映射回流形上的点。

#### 为什么潜在空间是连续的？

一个训练良好的自编码器的潜在空间具有以下性质：
- **连续性**：潜在空间中相邻的点解码后是相似的图像
- **可插值**：在潜在空间中两个点之间线性插值，解码后得到平滑过渡的图像

这就是为什么 VAE 可以从潜在空间随机采样生成新图像——它学会了整个流形，而不仅仅是训练样本点。


In [ ]:
# === 简单自编码器 (MNIST) ===
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class Autoencoder(nn.Module):  # 所有网络层的基类
    """简单全连接自编码器: 784 → 32 → 784"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(  # 顺序容器
            nn.Linear(784, 128),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(128, 64),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(64, 32),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(32, 784),  # 全连接层 y=Wx+b
            nn.Sigmoid()
        )
        self.decoder = nn.Sequential(  # 顺序容器
            nn.Linear(32, 64),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(64, 128),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(128, 784),  # 全连接层 y=Wx+b
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat.view(x.size(0), 784)  # unflatten to 28x28

# 训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder().to(device)  # 将数据移到 GPU 或 CPU
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    './data', train=True,
    download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")
print("训练中...")

history = []
for epoch in range(10):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    total_loss = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        x_hat, _ = model(x_batch)
        loss = criterion(x_hat, x_batch)
        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 用累积的梯度更新参数
        
        total_loss += loss.item()  # 取出单个数值
    
    history.append(loss.item())  # 取出单个数值
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}")  # 取出单个数值

# 重建效果
model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
with torch.no_grad():  # 禁用梯度计算（推理/评估时用）
    sample = next(iter(train_loader))[0].unsqueeze(0).to(device)  # 将数据移到 GPU 或 CPU
    recon = model(sample).view(28, 28)  # 改变张量形状（不拷贝数据）
    
fig, axes = plt.subplots(1, 3, figsize=(12, 5))  # 创建子图网格
# 原始
axes[0].imshow(train_dataset[0][0].view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[0].set_title('Original')
# 重建
axes[1].imshow(recon.cpu().view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[1].set_title(f'Reconstructed (epoch 10, loss={history[-1]:.4f})')
# 随机生成
z_random = torch.randn(1, 32).to(device)  # 标准正态分布随机张量
generated = model.decoder(z_random.view(1, 32, 1, 1))  # 改变张量形状（不拷贝数据）
axes[2].imshow(generated.cpu().view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[2].set_title('Generated from Random Latent Code')

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/autoencoder.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 10.4 变分自编码器 (VAE)：生成式模型的进化

VAE (Variational Autoencoder, 2013) 是自编码器的生成式扩展。它不满足于
"精确重建"，而是学习数据的**概率分布**——潜在空间中的每个点代表一个
概率分布，Decoder 输出的是**分布参数** $(\mu, \sigma)$。

### VAE 核心公式

$$

\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}
\left[\log p_\theta(\mathbf{x}|\mathbf{z})\right]}_{\text{重建误差}}
- \underbrace{D_{\text{KL}}\left(q_\phi(\mathbf{z}|\mathbf{x}) \|
p_\theta(\mathbf{z})\right)}_{\text{KL 正则化}}

$$

其中：
- **Encoder** $q_\phi(\mathbf{z}|\mathbf{x})$：从数据 $\mathbf{x}$ 推断潜在分布
- **Decoder** $p_\theta(\mathbf{x}|\mathbf{z})$：从潜在变量 $\mathbf{z}$ 生成数据
- 第一项是**重建损失**（希望 $\hat{\mathbf{x}} \approx \mathbf{x}$）
- 第二项是**KL 散度**（迫使近似后验 $q_\phi$ 接近先验 $p_\theta$）


In [ ]:
# === VAE 实现 (MNIST) ===
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
class VAE(nn.Module):  # 所有网络层的基类
    """简单 VAE: 784 → (32, 32) → 2 → 784"""
    def __init__(self):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(  # 顺序容器
            nn.Linear(784, 256),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(256, 128),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(128, 64),  # 全连接层 y=Wx+b
            nn.ReLU(),  # ReLU 激活函数
            nn.Linear(64, 2)  # 全连接层 y=Wx+b
        )
        # Mu, Logvar → latent z
        self.fc_mu = nn.Linear(64, 2)  # 全连接层 y=Wx+b
        self.fc_logvar = nn.Linear(64, 2)  # 全连接层 y=Wx+b
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar  # 返回结果
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z  # 返回结果
    
    def decode(self, z):
        h = self.decoder(z)
        return torch.sigmoid(h)  # 返回结果
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat  # 返回结果

model = VAE().to(device)  # 将数据移到 GPU 或 CPU
optimizer = optim.Adam(model.parameters(), lr=0.001)

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    './data', train=True,
    download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

history = []
for epoch in range(10):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    total_loss = 0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        mu, logvar = model.encode(x_batch)
        z = model.reparameterize(mu, logvar)
        x_hat = model.decode(z)
        
        loss_recon = F.mse_loss(x_hat, x_batch)
        loss_kl = F.kl_div(
            F.log_softmax(z, dim=1),
            F.log_softmax(model.reparameterize(mu, logvar), dim=1)
        )
        loss = loss_recon + 0.1 * loss_kl
        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 用累积的梯度更新参数
        
        total_loss += loss.item()  # 取出单个数值
    
    history.append(loss.item())  # 取出单个数值
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1}: recon_loss={loss_recon.item():.4f}, kl_loss={loss_kl.item():.4f}")  # 取出单个数值

# 生成与重建
model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
with torch.no_grad():  # 禁用梯度计算（推理/评估时用）
    z_random = torch.randn(1, 2).to(device)  # 标准正态分布随机张量
    mu, logvar = model.encode(torch.randn(64, 1, 28, 28).to(device))  # 标准正态分布随机张量
    sample = model.reparameterize(mu, logvar).view(1, 2, 28, 28)  # 改变张量形状（不拷贝数据）
    generated = model.decode(sample).view(1, 28, 28)  # 改变张量形状（不拷贝数据）

fig, axes = plt.subplots(2, 3, figsize=(14, 5))  # 创建子图网格
axes[0, 0].imshow(train_dataset[0][0].view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[0, 0].set_title('Original')
axes[0, 1].imshow(generated.cpu().view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[0, 1].set_title('Generated from VAE')
axes[1, 0].imshow(train_dataset[1][0].view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[1, 0].set_title('Original (sample 1)')
axes[1, 1].imshow(generated.cpu().view(28, 28), cmap='gray')  # 改变张量形状（不拷贝数据）
axes[1, 1].set_title('Generated (sample 1)')

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/vae_demo.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("\n关键洞察:")
print("  PCA: 全局线性结构 → 10 个数字类别混在一起")
print("  t-SNE: 局部邻域结构 → 清晰的 10 个聚类")
print("  VAE: 连续潜在空间 → 10 个数字的平滑流形")


# 第二部分：生成对抗网络 (GAN)

## 10.5 GAN 基础回顾

**生成对抗网络 (Generative Adversarial Network, GAN)** 由 Ian Goodfellow 于 2014 年提出，
开创了生成式建模的对抗训练范式。

### 核心思想：博弈论

GAN 由两个网络组成，进行**极小极大博弈 (Minimax Game)**：

- **生成器 (Generator, $G$)**：从随机噪声 $\mathbf{z} \sim p_z$ 生成假样本 $G(\mathbf{z})$，目标是"骗过"判别器
- **判别器 (Discriminator, $D$)**：判断输入是真实数据还是生成器伪造的，目标是"揪出"假样本

### GAN 的优化目标

$$

\min_G \max_D V(D, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}} [\log D(\mathbf{x})] + \mathbb{E}_{\mathbf{z} \sim p_z} [\log(1 - D(G(\mathbf{z})))]

$$

- 判别器 $D$ 希望最大化 $V$：对真实数据输出 1，对生成数据输出 0
- 生成器 $G$ 希望最小化 $V$：让 $D(G(z))$ 尽可能接近 1

### 训练交替进行

```
for each iteration:
    1. 训练判别器（固定生成器）：
        - 真实样本 x → D(x) 尽量接近 1
        - 生成样本 G(z) → D(G(z)) 尽量接近 0
    2. 训练生成器（固定判别器）：
        - 生成样本 G(z) → D(G(z)) 尽量接近 1
```

### GAN 的常见问题

- **模式坍塌 (Mode Collapse)**：生成器只产出少数几种样本，缺乏多样性
- **训练不稳定**：判别器和生成器的平衡很难维持
- **梯度消失**：当判别器太强时，生成器收不到有效梯度信号


## 10.6 DDcGAN：双判别器条件对抗网络

**DDcGAN (Dual-Discriminator Conditional GAN)** 是一种改进的条件 GAN 架构，
通过引入**两个结构不同的判别器**来解决 GAN 训练中的核心问题。

### 论文核心思想

传统 GAN 只有一个判别器，容易出现以下问题：
1. 判别器过于强大 → 生成器梯度消失
2. 判别器过于弱小 → 生成器得不到有效反馈
3. 单一判别器可能只关注某些特征（如纹理），忽略其他（如结构）

**DDcGAN 的解决方案：**

> 使用两个判别器——一个关注**全局结构**，一个关注**局部细节**。
> 两个判别器相互补充，迫使生成器同时优化全局和局部质量。

### 架构设计

```
           ┌──────────┐
           │  Noise z │ + [Condition c]
           └────┬─────┘
                ↓
        ┌───────────────┐
        │ Generator (G) │
        └───────┬───────┘
                ↓
          Generated x_fake
                │
        ┌───────┴───────┐
        ↓               ↓
   ┌─────────┐    ┌──────────┐
   │ D_global│    │ D_local  │
   │ (全局)  │    │ (局部)   │
   └────┬────┘    └─────┬────┘
        ↓               ↓
   "真实/假"      "真实/假"
        │               │
        └───────┬───────┘
        Combined Loss → Update G
```

### 两个判别器的角色

| 判别器 | 关注点 | 输入 | 架构特点 |
|--------|--------|------|---------|
| **$D_{\text{global}}$** | 整体结构、全局一致性 | 完整图像 | 较深网络，大感受野 |
| **$D_{\text{local}}$** | 局部细节、纹理质量 | 随机裁剪的图像块 | 较浅网络，局部特征 |

### 条件注入

DDcGAN 是**条件 GAN (cGAN)**——生成器和判别器都以条件 $c$（如类别标签）为额外输入：

$$

\text{生成器：} \quad G(\mathbf{z}, c) \rightarrow \hat{\mathbf{x}}

$$

$$

\text{判别器：} \quad D(\mathbf{x}, c) \rightarrow [0, 1]

$$


## 10.7 DDcGAN 损失函数详解

### 对抗损失 (Adversarial Loss)

DDcGAN 使用 **Hinge Loss** 替代原始 GAN 的交叉熵，训练更稳定：

**判别器损失：**

$$

\mathcal{L}_{D} = \mathbb{E}_{\mathbf{x}\sim p_{\text{real}}} [\max(0, 1 - D(\mathbf{x}))] + \mathbb{E}_{\mathbf{z}\sim p_z} [\max(0, 1 + D(G(\mathbf{z})))]

$$

直觉：
- $D(\mathbf{x}_{\text{real}})$ 应 $\ge 1$（大于 1 不惩罚）
- $D(\mathbf{x}_{\text{fake}})$ 应 $\le -1$（小于 -1 不惩罚）

**生成器损失：**

$$

\mathcal{L}_{G}^{\text{adv}} = -\mathbb{E}_{\mathbf{z}\sim p_z} [D(G(\mathbf{z}))]

$$

生成器希望 $D(G(\mathbf{z}))$ 尽可能大（被判别为真）。

### 双判别器联合损失

DDcGAN 将两个判别器的损失加权组合：

$$

\mathcal{L}_{D}^{\text{total}} = \mathcal{L}_{D_{\text{global}}} + \lambda \cdot \mathcal{L}_{D_{\text{local}}}

$$

$$

\mathcal{L}_{G}^{\text{total}} = \mathcal{L}_{G}^{\text{adv}} + \alpha \cdot \mathcal{L}_{\text{recon}}

$$

其中 $\mathcal{L}_{\text{recon}}$ 是可选的**重建损失 (Reconstruction Loss)**——在图像翻译等任务中使用 $L_1$ 距离。
系数 $\lambda$ 和 $\alpha$ 平衡不同损失项。典型设置：$\lambda = 0.5$，$\alpha = 10$（重建损失权重更大）。

### 为什么双判别器有效？

1. **多角度反馈**：全局判别器提供"形状对不对"的信号；局部判别器提供"纹理真不真"的信号
2. **难度自适应**：当全局判别器变强时，局部判别器可能还弱，生成器仍有梯度来源（反之亦然）
3. **防止模式坍塌**：两个判别器关注不同特征，生成器必须同时满足两者的要求 → 更多样的输出


In [ ]:
# === DDcGAN 核心组件实现（PyTorch）===
import torch
import torch.nn as nn
import torch.nn.functional as F

class Generator(nn.Module):  # 所有网络层的基类
    """DDcGAN 生成器：噪声 + 条件 → 图像
    使用转置卷积 (ConvTranspose2d) 从低分辨率逐步上采样
    """
    def __init__(self, latent_dim=100, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        # 条件嵌入：将类别标签映射到 latent 空间
        self.label_embedding = nn.Embedding(num_classes, latent_dim)  # 词嵌入层
        
        # 生成器主体：latent_dim*2 (noise + embed) → 4×4×512 → ... → 32×32×3
        self.main = nn.Sequential(  # 顺序容器
            # 输入: (latent_dim*2) × 1 × 1
            nn.ConvTranspose2d(latent_dim * 2, feature_dim * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_dim * 8),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (512) × 4 × 4
            
            nn.ConvTranspose2d(feature_dim * 8, feature_dim * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (256) × 8 × 8
            
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.ReLU(True),
            # 状态: (128) × 16 × 16
            
            nn.ConvTranspose2d(feature_dim * 2, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # 输出范围 [-1, 1]
            # 状态: (3) × 32 × 32
        )
    
    def forward(self, z, labels):
        # 条件嵌入
        c = self.label_embedding(labels).unsqueeze(-1).unsqueeze(-1)  # 增加一个维度
        # 拼接噪声和条件
        z = z.unsqueeze(-1).unsqueeze(-1)  # 增加一个维度
        z_c = torch.cat([z, c], dim=1)  # (batch, latent_dim*2, 1, 1)
        return self.main(z_c)  # 返回结果

# ========================================
class GlobalDiscriminator(nn.Module):  # 所有网络层的基类
    """全局判别器：关注整体图像结构
    使用较深的网络 + 大感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=64):
        super().__init__()
        
        # 条件嵌入（投影判别器风格）
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 8)  # 词嵌入层
        
        self.main = nn.Sequential(  # 顺序容器
            # 输入: 3 × 32 × 32
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),  # 二维卷积层
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 16 × 16
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 8 × 8
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (256) × 4 × 4
            
            nn.Conv2d(feature_dim * 4, feature_dim * 8, 4, 1, 0, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 8),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (512) × 1 × 1
        )
    
    def forward(self, img, labels):
        features = self.main(img)  # (batch, 512, 1, 1)
        features = features.view(features.size(0), -1)  # (batch, 512)
        
        # 投影判别器：内积形式注入条件
        label_embed = self.label_embedding(labels)  # (batch, 512)
        output = (features * label_embed).sum(dim=1, keepdim=True)  # (batch, 1)
        return output  # 返回结果

# ========================================
class LocalDiscriminator(nn.Module):  # 所有网络层的基类
    """局部判别器：关注随机图像块的纹理和细节
    使用较浅的网络 + 局部感受野
    """
    def __init__(self, num_classes=10, img_channels=3, feature_dim=32):
        super().__init__()
        
        self.label_embedding = nn.Embedding(num_classes, feature_dim * 4)  # 词嵌入层
        
        self.main = nn.Sequential(  # 顺序容器
            # 输入: 3 × 16 × 16 (从原图随机裁剪)
            nn.Conv2d(img_channels, feature_dim, 4, 2, 1, bias=False),  # 二维卷积层
            nn.LeakyReLU(0.2, inplace=True),
            # (32) × 8 × 8
            
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 2),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (64) × 4 × 4
            
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 1, 0, bias=False),  # 二维卷积层
            nn.BatchNorm2d(feature_dim * 4),  # 二维批归一化
            nn.LeakyReLU(0.2, inplace=True),
            # (128) × 1 × 1
        )
    
    def forward(self, img_patch, labels):
        features = self.main(img_patch)
        features = features.view(features.size(0), -1)  # 改变张量形状（不拷贝数据）
        label_embed = self.label_embedding(labels)
        output = (features * label_embed).sum(dim=1, keepdim=True)  # 沿指定维度求和
        return output  # 返回结果


In [ ]:
# === DDcGAN 损失函数 ===
import torch.nn.functional as F
class DDcGANLoss:
    """DDcGAN Hinge Loss + 双判别器加权"""
    
    def __init__(self, lambda_local=0.5):
        self.lambda_local = lambda_local
    
    def discriminator_loss(self, D_global_out_real, D_global_out_fake,
                            D_local_out_real, D_local_out_fake):
        """
        Hinge Loss for both discriminators
        D_real 应 >= 1, D_fake 应 <= -1
        """
        # 全局判别器
        loss_D_global_real = F.relu(1.0 - D_global_out_real).mean()  # 沿指定维度求均值
        loss_D_global_fake = F.relu(1.0 + D_global_out_fake).mean()  # 沿指定维度求均值
        loss_D_global = loss_D_global_real + loss_D_global_fake
        
        # 局部判别器
        loss_D_local_real = F.relu(1.0 - D_local_out_real).mean()  # 沿指定维度求均值
        loss_D_local_fake = F.relu(1.0 + D_local_out_fake).mean()  # 沿指定维度求均值
        loss_D_local = loss_D_local_real + loss_D_local_fake
        
        return loss_D_global + self.lambda_local * loss_D_local  # 返回结果
    
    def generator_loss(self, D_global_out_fake, D_local_out_fake):
        """生成器希望 D(G(z)) 尽可能大"""
        loss_G_global = -D_global_out_fake.mean()  # 沿指定维度求均值
        loss_G_local = -D_local_out_fake.mean()  # 沿指定维度求均值
        return loss_G_global + self.lambda_local * loss_G_local  # 返回结果


In [ ]:
# === DDcGAN 训练 Demo：在 MNIST 上生成手写数字 ===
# 使用简化版（灰度图 1 通道）训练少量 epoch 演示

import torch
import torch.optim as optim
def train_ddcgan_demo(epochs=10, batch_size=64, latent_dim=100, device='cpu'):
    """DDcGAN 在 MNIST 上的训练演示"""
    from torchvision import datasets, transforms
    
    # 数据加载
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # → [-1, 1]
    ])
    
    dataset = datasets.MNIST('./data', train=True,
                              download=True, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # 模型（灰度图 1 channel）
    G = Generator(latent_dim=latent_dim, num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    D_global = GlobalDiscriminator(num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    D_local = LocalDiscriminator(num_classes=10, img_channels=1).to(device)  # 将数据移到 GPU 或 CPU
    
    criterion = DDcGANLoss(lambda_local=0.5)
    
    # 优化器
    opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = optim.Adam(
        list(D_global.parameters()) + list(D_local.parameters()),
        lr=2e-4, betas=(0.5, 0.999)
    )
    
    history = {'G_loss': [], 'D_loss': []}
    
    for epoch in range(epochs):
        for i, (real_imgs, labels) in enumerate(dataloader):
            real_imgs, labels = real_imgs.to(device), labels.to(device)  # 将数据移到 GPU 或 CPU
            batch = real_imgs.size(0)
            
            # ---- 1. 训练判别器 ----
            opt_D.zero_grad()  # 清空上一轮的梯度
            
            # 生成假样本
            z = torch.randn(batch, latent_dim, device=device)  # 标准正态分布随机张量
            fake_imgs = G(z, labels).detach()  # detach: 不传播到 G
            
            # 获取局部 patch（随机裁剪 16×16）
            _, _, h, w = real_imgs.shape
            top = torch.randint(0, h - 16 + 1, (1,)).item() if h > 16 else 0  # 取出单个数值
            left = torch.randint(0, w - 16 + 1, (1,)).item() if w > 16 else 0  # 取出单个数值
            real_patches = real_imgs[:, :, top:top+16, left:left+16]
            fake_patches = fake_imgs[:, :, top:top+16, left:left+16]
            
            # 判别器输出
            Dg_real = D_global(real_imgs, labels)
            Dg_fake = D_global(fake_imgs, labels)
            Dl_real = D_local(real_patches, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_D = criterion.discriminator_loss(Dg_real, Dg_fake, Dl_real, Dl_fake)
            loss_D.backward()  # 反向传播，计算梯度
            opt_D.step()  # 用梯度更新参数
            
            # ---- 2. 训练生成器 ----
            opt_G.zero_grad()  # 清空上一轮的梯度
            
            z = torch.randn(batch, latent_dim, device=device)  # 标准正态分布随机张量
            fake_imgs = G(z, labels)
            
            # 局部 patch
            _, _, h, w = fake_imgs.shape
            top2 = torch.randint(0, max(h-16+1, 1), (1,)).item()  # 取出单个数值
            left2 = torch.randint(0, max(w-16+1, 1), (1,)).item()  # 取出单个数值
            fake_patches = fake_imgs[:, :, top2:top2+16, left2:left2+16]
            
            Dg_fake = D_global(fake_imgs, labels)
            Dl_fake = D_local(fake_patches, labels)
            
            loss_G = criterion.generator_loss(Dg_fake, Dl_fake)
            loss_G.backward()  # 反向传播，计算梯度
            opt_G.step()  # 用梯度更新参数
            
            if i % 150 == 0:
                history['D_loss'].append(loss_D.item())  # 取出单个数值
                history['G_loss'].append(loss_G.item())  # 取出单个数值
        
        print(f"Epoch {epoch+1:2d}/{epochs} | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")  # 取出单个数值
    
    return G, D_global, D_local, history  # 返回结果

# 运行训练（CPU 模式下较快完成）
device_train = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G_trained, _, _, history = train_ddcgan_demo(epochs=5, batch_size=64, device=device_train)


In [ ]:
# === DDcGAN 生成结果可视化 ===
import matplotlib.pyplot as plt
import torch
@torch.no_grad()
def generate_and_show(G, num_classes=10, samples_per_class=8, device='cpu'):
    G.eval()  # 切换评估模式
    latent_dim = G.latent_dim
    
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(samples_per_class, num_classes))  # 创建子图网格
    
    for cls in range(num_classes):
        z = torch.randn(samples_per_class, latent_dim, device=device)  # 标准正态分布随机张量
        labels = torch.full((samples_per_class,), cls, dtype=torch.long, device=device)
        fake_imgs = G(z, labels).cpu()  # 移到 CPU
        
        for j in range(samples_per_class):
            img = fake_imgs[j].squeeze().numpy()  # 去除大小为 1 的维度
            img = (img + 1) / 2  # [-1,1] → [0,1]
            axes[cls, j].imshow(img, cmap='gray')
            axes[cls, j].axis('off')
            if j == 0:
                axes[cls, j].set_ylabel(f'Class {cls}', fontsize=10, rotation=0, labelpad=20)
    
    plt.suptitle('DDcGAN Generated MNIST Digits (5 epochs)', fontsize=14, y=1.02)
    plt.tight_layout()  # 自动调整子图间距
    plt.savefig('../fig/ddcgan_generated.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
    plt.show()  # 显示图表

generate_and_show(G_trained, device=device_train)

# 训练曲线
if history['G_loss']:
    fig, ax = plt.subplots(figsize=(8, 4))  # 创建子图网格
    steps = range(len(history['G_loss']))
    ax.plot(steps, history['D_loss'], label='D Loss', alpha=0.8)
    ax.plot(steps, history['G_loss'], label='G Loss', alpha=0.8)
    ax.set_xlabel('Training Steps (×150)'); ax.set_ylabel('Loss')
    ax.set_title('DDcGAN Training Curves (Hinge Loss)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()  # 自动调整子图间距
    plt.savefig('../fig/ddcgan_training.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
    plt.show()  # 显示图表


## 10.8 DDcGAN 关键设计要点

### 1. 投影判别器 (Projection Discriminator)

DDcGAN 使用**投影判别器**来注入条件信息，而非简单拼接：

$$

D(\mathbf{x}, c) = \mathbf{w}_c^T \phi(\mathbf{x}) + \psi(\phi(\mathbf{x}))

$$

其中 $\phi(\mathbf{x})$ 是判别器对图像提取的特征，$\mathbf{w}_c$ 是类别 $c$ 的嵌入向量。
这种设计已被证明优于简单的条件拼接。

### 2. 局部判别器的随机裁剪

每次前向传播时从图像中**随机裁剪**一个固定大小的 patch（如 16×16），
迫使局部判别器学会评估任意位置的纹理质量。

### 3. 架构不对称设计

| 组件 | 全局判别器 | 局部判别器 |
|------|-----------|-----------|
| 网络深度 | 4 层卷积 | 3 层卷积 |
| 特征维度 | 64 起始 → 512 最终 | 32 起始 → 128 最终 |
| 感受野 | 覆盖全图 (32×32) | 仅覆盖 patch (16×16) |
| 功能 | 判断结构一致性 | 判断纹理真实性 |

这种不对称设计是关键——两个判别器学到的特征**互补而非冗余**。


## 本章总结

### 降维方法对比

| 方法 | 类型 | 一句话总结 |
|------|------|-----------|
| **PCA** | 线性降维 | 见第九章——最大方差方向的线性投影 |
| **t-SNE** | 非线性降维 | PCA 看不到的非线性聚类结构 |
| **UMAP** | 非线性降维 | 比 t-SNE 更快更稳定的替代方案 |

### 生成模型进化

| 方法 | 关键创新 | 局限 |
|------|---------|------|
| **Autoencoder** | 压缩-重建，学习紧凑表示 | 潜在空间不连续，无法采样生成 |
| **VAE** | 概率编码，reparameterization trick | 生成图像偏模糊 |
| **GAN** | 对抗训练，博弈论视角 | 训练不稳定，模式坍塌 |
| **DDcGAN** | 双判别器（全局+局部），条件投影 | 计算开销更高 |

### DDcGAN 要点

| 概念 | 关键设计 |
|------|---------|
| 双判别器 | 全局判别器（整体合理性）+ 局部判别器（细节质量） |
| 条件注入 | 投影判别器 (x,c) = w_c^T \phi(x)$ 而非简单拼接 |
| 损失函数 | Hinge Loss 替代交叉熵，训练更稳定 |
| 不对称设计 | 两个判别器结构不同（全局 vs 局部），防止同质化 |
